<center><img src="https://i.pinimg.com/736x/17/e5/cf/17e5cfabf63bb3f40464ae6fc6acc445.jpg" width="1200" height="300"></center>

## <b>1 <span style='color:#F1A424'>|</span>네이버 영화 댓글 감성분석 실습</b> 

<div style="color:white;display:fill;border-radius:8px;font-size:100%; letter-spacing:1.0px;"><p style="padding: 5px;color:white;text-align:left;"><b><span style='color:#F1A424'>WHAT WE WILL DO IN THIS SECTION</span></b></p></div>

- 날짜 : 2026-06-10(수요일)_오전수업
- 핵심 내용 : 
- 사용할 데이터셋 : `NSMC(NAVER Sentiment movei corpus)`
    - 영화에 대한 데이터셋 ➔ 그외 도메인에 대한 분석력은 떨어짐. <br> 따라서 데이터셋을 수집할 때 목적이 무엇인가에 집중해야함. 
- **입력** : 영화에 대한 댓글 
- **출력** : 긍정(1)😊/부정(0)😡
- 기능 : 이진분류, 해당 댓글이 긍정적인 댓글인가, 부정적인 댓글인가? <br> ➔ 해당 이진분류의 관점에서 긍정이 아닌 것이 부정적인가로 보게될수 없음. <br> 부정 or 감성이 드러나지 않는 케이스일 수도있다. 
- 중요도 : 사람들의 리뷰를 수집해서 긍정과 부정을 판단 ➔ `감성분석` 이라고 함. 또는 중립이 들어가기도 함. 
- 조합 : 부정 중에서도 슬픔, 화남, .. 여러 감정적인 분류도 가능함. 즉 사람의 감성을 분석

<br>

<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.1 | </span></span></b>nsmc 데이터셋 로드 및 DF 변환</b></p></div>
깃허브에 어떤 파일이 있을 때, 혹은 데이터나 코드 ➔ raw를 누르면 경로가 나옴
이걸 request.get해서 받아올 수 잇음.

1. nsmc 깃헙 들어가기 (https://github.com/e9t/nsmc)
2. 데이터셋 파일에 raw 누르고 주소 카피 <br> (https://raw.githubusercontent.com/e9t/nsmc/refs/heads/master/ratings.txt)
3. pd.read_csv(resp.text): 경로로 읽어버림 ➔ 오류 발생 <br> 해당 상황에서 입력을 할 수 있는 stringIO 객체를 넣어줘야함

In [1]:
''' 데이터셋 다운로드 및 dataframe 생성 '''
import pandas as pd
import requests
from io import StringIO

url = "https://raw.githubusercontent.com/e9t/nsmc/refs/heads/master/ratings.txt"
resp = requests.get(url)

#응답 정보 저장
if resp.status_code == 200: # 요청 성공했어요
    str_io = StringIO(resp.text) # resp.text랑 연결하는 입출력 객체를 만들어라

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">🤔 io (INPUT / OUTPUT)  </mark>** 
<br> 외부 자원(파일, DB, 원격컴)과 연결 메모리에 STR 값이 있을 때 사용하는 게 StringIO. 함수를 호출하려고 할때 함수가 문자가 아니라 입출력을 하기 위한 객체를 받아야하는 경우가 있음. 이경우는 문자열로는 안되고 io객체로 제공해야함.

- **다음과 같은 상황에 사용**
    - 테스트용 데이터를 문자열로 만들어서 파일처럼 사용
    - 실제 디스크에 파일을 저장하지 않고 메모리에서만 파일 작업을 하고 싶을 때 사용 <br> 속도 빠름
    - 문자열 데이터를 파일 형태로 변환: 웹에서 받은 CSV 데이터나 설정 파일 내용을 문자열로 받았을 때, 이를 파일로 저장하지 않고 바로 파일을 읽는 라이브러리에 전달할 수 있다.


In [2]:
df = pd.read_csv(str_io, sep ="\t") #보통 csv는 쉼표, 여기는 탭으로 연결됨
df.shape

(200000, 3)

In [3]:
df.head()

,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.2 | 
</span></span></b>데이터셋 전처리 및 EDA</b></p></div>


**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">전처리 1단계 : document 컬럼 결측치 처리</mark>** 

그 양이 미미함. ➔ 결측치 제거

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        200000 non-null  int64
 1   document  199992 non-null  str  
 2   label     200000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 21.1 MB


In [ ]:
# 결측치 제거하기 
df = df.dropna()
df.isnull().sum()

id          0
document    0
label       0
dtype: int64

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">전처리 2단계 : 중복 행(documnet) 확인/ 제거</mark>** 

똑같은 내용의(documnet)가 같은 중복 행이 존재하는가? 확인
- 해석 :
    - 느낌표 의 경우 라벨이 0과 1 로 나뉨 ➔ 노이즈가 될 수 있음
    - `keep = FALSE` 로 ➔ 모든 중복 행들을 TRUE로 반환
    - `keep = "first" | "last"` : 
        - 중복된 것들에서 first 는 처음 행
        - last는 마지막 행만 반환
        - 중복된 값만 확인하고 싶을 때 한 행만 출력함.

In [10]:
df[df.duplicated(subset=['document'],keep=False)].sort_values("document")

,id,document,label
12986,181912,!,1
136580,6993402,!,0
15611,7868198,",",1
59493,7448690,",",1
31251,2853697,",,,",1
...,...,...,...
73595,5153363,힐러리 더프의 매력에 빠지다!!!,1
104378,3010576,힘내세요,0
152847,4052413,힘내세요,0
118102,7024515,힘들다,0


In [11]:
# 중복행 삭제 - 하나만 남기고 나머지 행은 삭제
df = df.drop_duplicates(subset=['document'])

In [12]:
df[df.duplicated(subset=['document'],keep=False)].sort_values("document")

,id,document,label


In [13]:
df.shape

(194543, 3)

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">전처리 3단계 : 인덱스 네임 초기화 </mark>** 
- 인덱스가 자동증가인 경우 ( not id ) ➔ 중복행 제거 후 초기화 추천

In [14]:
#인덱스 네임 초기화
df.reset_index(drop=True, inplace= True)

<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.3 | </span></span></b>데이터셋을 제공하는 class 정의 | 모듈화하기</b></p></div> 

1. 객체 생성 및 초기화 `__init__`
    - 기능 요약 :
        - 파이프라인 전체를 총괄
        - 최종결과물(self.nsmc_df)을 속성으로 보관 
    - data/ratings.txt 경로 지정 및 Kiwi 객체 생성
    - load_nsmc_dataset() 호출

2. 데이터 다운로드 및 저장 `load_nsmc_dataset`
    - try : 로컬에 파일(data/ratings.txt)이 있는지 확인
    - except : 없다면 GitHub에서 원본 다운로드 후 저장 및 로드
    - **output : pd.DataFrame (원본)**

3. 전체 데이터 전처리 `preprocess`
    - **input : pd.DataFrame (원본)**
    - 결측치(NaN) 제거
    - 중복 행(댓글) 제거
    - 각 댓글에 tokenize() 함수 일괄 적용(apply)
    - 토큰화 결과 빈 문자열이 된 행 제거
    - 최종 데이터프레임을 `self.nsmc_df`에 저장
    - **output : pd.DataFrame (정제 완료)**


3. 형태소 기반 개별 텍스트 토큰화 진행 `tokenize`
    - kiwi 띄어쓰기 교정(space)
    - 형태소 분석 후 원형(lemma) 추출해서
    - 공백 (" ")으로 구분된 `하나의 문자열`로 결합 후 return

In [6]:
import os
import requests
import pandas as pd
from kiwipiepy import Kiwi

os.makedirs("data", exist_ok=True)

class NSMCTokenizer:
    
    def  __init__ (self):
        # 1. 데이터셋을 저장할 파일 경로 지정
        # os.makedirs("data", exist_ok=True)
        self.data_file_path = "data/ratings.txt"
        
        # 2. 토크나이저 키위 객체 생성
        self.kiwi = Kiwi()
        
        # 3. 토큰화가 끝난 dataset의 DataFrame을 속성으로 저장
        ## step1. dataset 다운로드 & df로 반환받기
        df = self.load_nsmc_dataset()
        
        ## step2. 댓글 전처리해서 토큰화.
        self.nsmc_df = self.preprocess(df)
        pass
    
    def load_nsmc_dataset(self) -> pd.DataFrame:
        '''
        nsmc 데이터셋을 다운받아서 (github_raw_url) data/ratings.txt로 저장
        dataframe으로 만들어서 반환
        ---
        args: 
        returns: 
            pd.DataFrame : 네이버 영화 댓글 데이터셋.
                - columns : id(댓글번호), document(댓글 내용), 
                            label(0:부정, 1:긍정)
        Raise: 
            Exception : 데이터프레임 생성과 데이터셋 다운로드 실패 시 발생.
        
        '''  
        
        try:
            df = pd.read_csv(self.data_file_path, sep='/t', encoding="utf-8")
        except:
            
            # 다운로드 ➔ 저장 ➔ df 생성
            url = "https://raw.githubusercontent.com/e9t/nsmc/refs/heads/master/ratings.txt"
            resp = requests.get(url)
            if resp.status_code == 200:
                # 다운로드 및 저장
                with open(self.data_file_path, "wt", encoding="utf-8") as f:
                    f.write(resp.text)
                # 데이터 프레임 생성
                df = pd.read_csv(self.data_file_path, sep='\t', encoding="utf-8")
            else:
                # 예외를 발생
                raise Exception( f"데이터파일을 다운받지 못했습니다. 에러코드 :{resp.status_code}")
            
        return df 
    def preprocess(self, df:pd.DataFrame) -> pd.DataFrame:
        '''
        전체 dataset을 전처리
        Step1.결측치 제거
        Step2. 중복(document)행 제거
        Step3. 개별 데이터 토큰화 작업
        
        Args: 
            df(pd.DataFrame) - 전처리 및 토큰화 대상 데이터셋
        Returns:
            pd.DataFrame - 처리 결과
        '''  
        result_df = df.dropna()
        # document 컬럼 중복행 제거 
        result_df = result_df.drop_duplicates(subset=['document'])
        
        # document(댓글)컬럼의 값들을 토큰화 처리 *apply 원소별로 일괄처리.
        result_df['document'] = result_df['document'].apply(self.tokenize)
        
        # document의 글자수가 0인 행들을 제거.(불용어 제거, 전처리 후 글자수가 0이 될 수 있음.)
        drop_row_index = result_df[result_df['document'].str.len() == 0].index#result_df의 데이터를 일괄처리해주는 
        result_df = result_df.drop(index=drop_row_index)
        
        return result_df         
    
    def tokenize(self, doc:str) -> str:
        '''
        Step1.하나의 댓글을 받아서 전처리(띄어쓰기 교정)한 뒤 토큰화
        Step 2. 하나의 문자열로 반환
        Arg :
            doc (str) - 토큰화할 댓글(document)
        
        Return : 
            str - 처리 결과 (띄어쓰기 교정, 토큰화)  
        '''
        doc = self.kiwi.space(doc)
        token_list = [] #토큰(원형)들 저장
        try:
            for token in self.kiwi.tokenize(doc): # list[token]
                token_list.append(token.lemma)
        except: 
            pass
        
        # token list를 문자열로 반환
        return " ".join(token_list)        

In [7]:
import time
s= time.time()

nsmc = NSMCTokenizer()

print(f"걸린 시간 : {time.time() - s}초")


C:\Users\Playdata\AppData\Local\Temp\ipykernel_3468\1800013782.py:42: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(self.data_file_path, sep='/t', encoding="utf-8")


걸린 시간 : 162.73426413536072초


In [8]:
print(nsmc.nsmc_df.shape)
nsmc.nsmc_df.head()

(194543, 3)


,id,document,label
0,8112052,어리다 ᆯ 때 보다 고 지금 다시 보다 어도 재밌다 어요 ㅋㅋ,1
1,8132799,"디자인 을 배우다 는 학생 으로 , 외국 디자이너 와 그 들 이 일구다 ᆫ 전통 을...",1
2,4655635,폴리스 스토리 시리즈 는 1 부터 뉴 까지 버리다 ᆯ께 하나 도 없다 음 .. 최고 .,1
3,9251303,와 .. 연기 가 진짜 개 쩔다 구나 .. 지루 하 ᆯ 거 이다 라고 생각 하 었 ...,1
4,10067386,안개 자욱 하 ᆫ 밤하늘 에 뜨다 어 있다 는 초승달 같다 은 영화 .,1


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.4 | </span></span></b> 모델링을 시작해보자</b></p></div>


**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">모델링 1단계 : x, y 데이터셋 스플릿 </mark>** 

In [27]:
X = nsmc.nsmc_df['document']
y = nsmc.nsmc_df['label']

print(X.shape, y.shape)
print("-" * 50)
print(y.value_counts())

(194543,) (194543,)
--------------------------------------------------
label
0    97277
1    97266
Name: count, dtype: int64


**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">모델링 2단계 : Train / Test / validation set으로 분리 </mark>** 

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# 2차 분리
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

In [29]:
print("X_train 개수:", len(X_train))
print("y_train 개수:", len(y_train))

X_train 개수: 124507
y_train 개수: 124507


**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">모델링 3단계 : 모델 파이프라인 구축  </mark>** 
> from sklearn.pipeline import Pipeline
- 전처리 : TfidfVectorizer
- 모델 : LogisticRegression

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

tfidf = TfidfVectorizer() #tuning: ngram_range, max_df, min_df
model = LogisticRegression()
steps = [
    ("tfidf", tfidf), ("model",model)
]

pipeline = Pipeline(steps=steps, verbose= True)

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">모델링 4단계 : 모델 학습 </mark>** 

In [31]:
print("X_train의 타입:", type(X_train), "크기:", len(X_train))
print("y_train의 타입:", type(y_train), "크기:", len(y_train))

X_train의 타입: <class 'pandas.Series'> 크기: 124507
y_train의 타입: <class 'pandas.Series'> 크기: 124507


In [33]:
import sklearn

# 주피터 노트북에서 파이프라인을 HTML 다이어그램 대신 텍스트(문자열)로 표현하도록 설정
sklearn.set_config(display="text")

In [34]:
pipeline.fit(X_train, y_train)

[Pipeline] ............. (step 1 of 2) Processing tfidf, total=   1.1s
[Pipeline] ............. (step 2 of 2) Processing model, total=   0.6s


Pipeline(steps=[('tfidf', TfidfVectorizer()), ('model', LogisticRegression())],
         verbose=True)

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">모델링 5단계 : 모델 검증 및 평가</mark>** 

In [35]:
pred_train = pipeline.predict(X_train)
pred_val = pipeline.predict(X_val)
print(
    accuracy_score(y_train, pred_train),
    accuracy_score(y_val, pred_val)
)

0.8633811753556025 0.834805795611527


In [36]:
pipeline.score(X_train,y_train)

0.8633811753556025

In [37]:
pipeline.score(X_test,y_test)

0.8300650235164101

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">모델링 6단계 : 새로운 데이터를 추론</mark>** 



In [ ]:
# 추론용 DEF 정의
def predict(*comment):
    #NSMC 를 이용해서 token화
    pre_comment = [nsmc.tokenize(doc) for doc in comment]
    pred = pipeline.predict(pre_comment)
    return pred

In [ ]:
result = predict(
    "이영화 진짜 별로다.",
    "시간 때우려고 봤는데 엄청 재밌다.",
    "배우들 연기 미쳤다.",
    "내 소중한 시간 돌려줘",
    "여기 식당 인테리어가 별로예요.",
    "봉스토랑 삼겹살조림이 감칠맛이 죽여줘요",)

In [49]:
import numpy as np
idx_to_class = np.array(['부정적 댓글', '긍정적 댓글'])
print(result, idx_to_class[result])

[0 1 0 0 0 0 1] ['부정적 댓글' '긍정적 댓글' '부정적 댓글' '부정적 댓글' '부정적 댓글' '부정적 댓글' '긍정적 댓글']


In [50]:
vocab = tfidf.get_feature_names_out()
vocab.shape

(38921,)

In [54]:
v = tfidf.transform(["이 영화 존나 별로다"])
v.toarray()

print(v)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 38921)>
  Coords	Values
  (0, 24719)	0.30123546809440144
  (0, 30530)	0.9535497851512248


2개의 인덱스 빼고는 다 0. 굉장히 SPARSE함. 
데이터의 피쳐가 단어일 뿐. 
피쳐엔지니어링하며 배운 것들을 모두 적용 가능함.
컬럼이 너무 많아지니까 PCA , 피쳐스케일링 등을 통해서 성능을 높일 수 있음.
